In [2]:
import pandas as pd
import re
from calendar import month_name

# ---- CONFIG ----
INPUT_FILE = "CoreAdvance CPI_matches.xlsx"
OUTPUT_FILE = "CoreAdvance Final CPI Output.xlsx"

# Load input Excel
df = pd.read_excel(INPUT_FILE)

# Trim last 4 chars of Filename
df["Filename"] = df["Filename"].astype(str).str[:-4]

def trim_leading_parenthetical(text):
    if pd.isna(text):
        return text

    s = str(text).lstrip()

    # Only trim if the text starts with "("
    if s.startswith("("):
        close_idx = s.find(")")
        if close_idx != -1:
            return s[close_idx + 1 :].strip()

    return s

# Apply trimming directly to input dataframe
df["CPI Snippets (LLM)"] = df["CPI Snippets (LLM)"].apply(trim_leading_parenthetical)

# -----------------------
# Helper functions
# -----------------------

# Client Name: all ALL-CAPS words, treat '-' as separator, exclude PDF
def extract_client_name_all_caps(filename):
    if pd.isna(filename):
        return ""
    text = str(filename)

    # Split on whitespace, then split hyphen-separated subparts
    parts = []
    for chunk in text.split():
        subparts = chunk.split('-')
        parts.extend(subparts)

    # Keep only tokens that are fully uppercase and contain at least one letter, and are not "PDF"
    caps = [p for p in parts if any(c.isalpha() for c in p) and p == p.upper() and p.upper() != "PDF"]
    return " ".join(caps).strip()

# Extract year (YYYY) from Fee Increase Effective Date(s)
# 1) Try to find a 4-digit year (20XX)
# 2) If not found, try to find MM/DD/YY or MM/DD/YYYY and convert to 4-digit year
def extract_year(text):
    if pd.isna(text):
        return ""
    s = str(text)

    # 1) look for explicit 4-digit year like 20XX
    match = re.search(r"\b(20\d{2})\b", s)
    if match:
        return match.group(1)

    # 2) look for MM/DD/YY or MM/DD/YYYY patterns
    m = re.search(r"\b(\d{1,2})/(\d{1,2})/(\d{2,4})\b", s)
    if m:
        year_part = m.group(3)
        if len(year_part) == 2:
            # assume 20YY for two-digit years
            year_full = 2000 + int(year_part)
        else:
            year_full = int(year_part)
        return str(year_full)

    return ""

# Extract month name from Fee Increase Effective Date(s)
# 1) Try full month name
# 2) If not found, parse MM/DD/YY or MM/DD/YYYY and map month number to month name
MONTHS = [m for m in month_name if m]  # month_name[1] = 'January' ... skip index 0
def extract_month(text):
    if pd.isna(text):
        return ""
    s = str(text)

    # 1) check for full month names (case-insensitive)
    for m in MONTHS:
        if m.lower() in s.lower():
            return m

    # 2) check for MM/DD/YY or MM/DD/YYYY
    m = re.search(r"\b(\d{1,2})/(\d{1,2})/(\d{2,4})\b", s)
    if m:
        month_num = int(m.group(1))
        if 1 <= month_num <= 12:
            return month_name[month_num]
    return ""

# Extract year from Contract Effective Date using last 4 characters and add 1
def extract_contract_effective_year_plus_one(text):
    if pd.isna(text):
        return ""
    s = str(text).strip()
    if len(s) < 4:
        return ""
    last4 = s[-4:]
    if last4.isdigit():
        return str(int(last4) + 1)
    return ""

# Compute CPI Terms (per Contract)
def compute_cpi_terms(row):
    min_fee = row.get("Minimum Fee Increase(s)")
    snippet = row.get("CPI Snippets (LLM)")

    # Normalize min_fee early
    if pd.isna(min_fee):
        min_fee = ""

    if pd.isna(snippet) or str(snippet).strip() == "":
        return "CPI Language not Found"

    snippet_str = str(snippet)

    # 🔹 If min_fee is blank, try extracting percentage from snippet
    #if min_fee == "":
    pct_matches = re.findall(
        r"(\d+(?:\.\d+)?)\s*(%|percent)",
        snippet_str,
        flags=re.IGNORECASE
    )

    if pct_matches:
        # Normalize all to % (e.g., "5 percent" -> "5%")
        percents = [f"{m[0]}%" for m in pct_matches]

        # Create comma-separated list
        min_fee = ", ".join(percents)

    else:
        min_fee = "NA"

    if "whichever is greater" in str(snippet).lower():
        return f"> CPI-U or {min_fee}"

    if ("cpi" not in str(snippet).lower()) and ("consumer price index" not in str(snippet).lower()):
        return f"Max {min_fee}"

    if min_fee == "NA":
        return "Limited to CPI-U"

    return f"< CPI-U or {min_fee}"

# Notice requirement based on CPI snippet presence
def compute_notice(snippet):
    if pd.isna(snippet) or str(snippet).strip() == "":
        return ""
    return "30 Days"

# Split Contract Language into normal vs For Review depending on presence of "30"
def split_contract_language(snippet):
    if pd.isna(snippet) or str(snippet).strip() == "":
        return "", ""
    snippet_str = str(snippet)
    if "30" in snippet_str:
        return snippet_str, ""
    else:
        return "", snippet_str

# Specific Contract Language/Information:
# - If "limited" appears (case-insensitive), return substring starting at last occurrence of "limited".
# - Else if snippet contains "30", return the entire snippet.
# - Else return blank.
def extract_specific_contract_language(snippet):
    if pd.isna(snippet) or str(snippet).strip() == "":
        return ""
    s = str(snippet)
    lower = s.lower()
    idx = lower.rfind("limited")
    if idx != -1:
        return s[idx:].strip()
    if "30" in s:
        return s.strip()
    return ""

# -----------------------
# Build output dataframe in required column order
# -----------------------
output = pd.DataFrame()

# 1. Client Name (all ALL-CAPS words in Filename in order, hyphen-separated tokens treated separately, exclude PDF)
output["Client Name"] = df["Filename"].apply(extract_client_name_all_caps)

# 2. Core
output["Core"] = "CoreAdvance"

# 3. Contract Type
output["Contract Type"] = df["Contract Type"]

# 4. Contract Effective Date
output["Contract Effective Date"] = df["Contract Effective Date"]

# 🔹 UPDATED: replace _ with -
output["Contract Effective Date"] = (
    df["Contract Effective Date"]
    .astype(str)
    .str.replace("_", "-", regex=False)
)

# 5. CPI Terms (per Contract)
output["CPI Terms (per Contract)"] = df.apply(compute_cpi_terms, axis=1)

# 6. CPI Eligibility Year
def compute_elig_year(row):
    snippet = row.get("CPI Snippets (LLM)")
    if pd.isna(snippet) or str(snippet).strip() == "":
        return ""

    # 1) Try Fee Increase Effective Date(s) for a 4-digit year OR MM/DD/YY
    fee_year = extract_year(row.get("Fee Increase Effective Date(s)"))
    if fee_year:
        return fee_year

    # 2) Fallback: Contract Effective Date (last 4 chars) + 1
    return extract_contract_effective_year_plus_one(row.get("Contract Effective Date"))

output["CPI Eligibility Year"] = df.apply(compute_elig_year, axis=1)

# 7. CPI Eligibility Month
# Try month name in Fee Increase Effective Date(s), else extract from MM/DD/YY if present
output["CPI Eligibility Month"] = df["Fee Increase Effective Date(s)"].apply(extract_month)

# 8. Notice Requirement
output["Notice Requirement"] = df["CPI Snippets (LLM)"].apply(compute_notice)

# NEW COLUMN: Specific Contract Language/Information (just before Contract Language/Information)
output["Specific Contract Language/Information"] = df["CPI Snippets (LLM)"].apply(extract_specific_contract_language)

# 9 & 10. Contract Language/Information and Contract Language/Information (For Review)
normal_col = []
review_col = []
for s in df["CPI Snippets (LLM)"]:
    normal, review = split_contract_language(s)
    normal_col.append(normal)
    review_col.append(review)

output["Contract Language/Information"] = normal_col
output["Contract Language/Information (For Review)"] = review_col

# 11. Item Type
output["Item Type"] = "Item"

# 12. Path
output["Path"] = "sites/fss/CPI/Lists/CU CPI Database"

output["OCR"] = df["OCR"]

output["Page Number"] = df["Page Number"]

# -----------------------
# Save and show result
# -----------------------
print(output.head(30))
output.to_excel(OUTPUT_FILE, index=False)
print(f"\nSaved to {OUTPUT_FILE}")


                          Client Name         Core     Contract Type  \
0                        FARMERS BANK  CoreAdvance         Amendment   
1                        FARMERS BANK  CoreAdvance         Amendment   
2                        FARMERS BANK  CoreAdvance         Amendment   
3                        FARMERS BANK  CoreAdvance         Amendment   
4                        FARMERS BANK  CoreAdvance         Amendment   
5                        FARMERS BANK  CoreAdvance         Amendment   
6                        FARMERS BANK  CoreAdvance         Amendment   
7                        FARMERS BANK  CoreAdvance         Amendment   
8                        FARMERS BANK  CoreAdvance         Amendment   
9                        FARMERS BANK  CoreAdvance         Amendment   
10                       FARMERS BANK  CoreAdvance         Amendment   
11                       FARMERS BANK  CoreAdvance  Master Agreement   
12                       FARMERS BANK  CoreAdvance  Master Agree